In [0]:
# Databricks notebook source
class Endpoints:
    def __init__(self):
        self.chat = "llm/v1/chat"
        self.embeddings = "llm/v1/embeddings"

    def create(self, name, model, provider="openai", task="chat"):
        from mlflow.deployments import get_deploy_client

        dp_client = get_deploy_client("databricks")
        ep_list = dp_client.list_endpoints()
        if name not in [ep["name"] for ep in ep_list]:
            print(f"Creating Model Endpoint...", end="")
            endpoint = dp_client.create_endpoint(
                config={
                    "name": name,
                    "config": {
                        "served_entities": [
                            {
                                "external_model": {
                                    "name": model,
                                    "provider": provider,
                                    "task": (
                                        self.chat if task == "chat" else self.embeddings
                                    ),
                                    "openai_config": {
                                        "openai_api_type": "openai",
                                        "openai_api_key_plaintext": "",
                                    },
                                },
                            }
                        ]
                    },
                    "ai_gateway": {"usage_tracking_config": {"enabled": True}},
                }
            )
            print("Done")

    def delete(self, name):
        from mlflow.deployments import get_deploy_client

        dp_client = get_deploy_client("databricks")
        ep_list = dp_client.list_endpoints()
        if name in [ep["name"] for ep in ep_list]:
            print(f"Deleting Model Endpoint...", end="")
            dp_client.delete_endpoint(endpoint=name)
            print("Done")